In [1]:
# STANDIN A0
# Load the data, the split and the shared helpers (parts/00_setup.ipynb). Dropped from the hand-in
# notebook, where the setup cells come first.
%run 00_setup.ipynb

Python 3.13.7, scikit-learn 1.6.1, numpy 2.5.3, pandas 3.0.6
Working directory: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project
Data folder: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project/UNSW-NB15
score() and score_proba() ready


Rows with an infinite or missing value: 0
Loaded 447,915 flows x 76 features


Raw rows:                      447,915
Exact duplicates removed:      141,742
Conflicting rows removed:        1,068
Clean rows (all are used):     305,105


Dropped 9 constant columns: Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, URG Flag Count, CWR Flag Count, ECE Flag Count, Fwd Bytes/Bulk Avg, Fwd Packet/Bulk Avg, Fwd Bulk Rate Avg
67 features: 65 continuous, 2 0/1 columns (Fwd PSH Flags, RST Flag Count)
TUNE_IDX: 36,612 training rows for settings searches
Setup complete: 22 shared names ready in 54 s


<!-- STEP A1 -->
## Part A: the security problem

**What is detected.** We build a network intrusion detector: for every network flow (one conversation
between two computers, summarised as 67 numbers about packet counts, sizes and timing) it decides whether
the flow is **normal traffic or part of an attack**, such as exploiting a software bug, fuzzing a service
with malformed data, scanning the network, a denial-of-service flood, a backdoor or a worm.

**Who uses it.** The analysts of a Security Operations Centre (SOC), who receive the detector's alerts
and must decide for each one whether to investigate, block the connection or ignore it.

**What a false alarm costs.** Every false alarm takes an analyst's time to check and dismiss. A network
sees millions of flows a day, so even a false alarm rate of 1% means about 10,000 false alarms per million
normal flows. Analysts then start ignoring alerts, and real attacks drown among the false ones (*alert
fatigue*). A **missed attack** costs more per case: the attacker gets in unnoticed and can steal data or
damage systems. The detector therefore has to catch most attacks (high recall) while keeping the false
alarm rate (FAR) low, and we report both.

**Data.** CIC-UNSW-NB15: the UNSW-NB15 traffic recorded in 2015 at UNSW Canberra (normal traffic plus
nine attack types), turned into flows by CICFlowMeter. After removing duplicates and conflicting rows we
use **all 305,105 clean flows** (26.3% attacks), with no subsample.

<!-- STEP A4 -->
## Part A: the trivial baseline

The simplest possible "detector" never raises an alarm: it calls **every flow benign**, the majority
class. It needs no learning, yet scores an accuracy equal to the share of benign flows (about 74%).
That is why accuracy alone says little here: **every model below must beat this line**, and the scores
that matter are the ones this baseline fails on:

- **recall = 0**: it catches no attack at all;
- **macro-F1 ≈ 0.42**: perfect-looking on the benign class, zero on the attack class;
- **PR-AUC = the attack share (≈ 0.26)**: what a model gets by guessing, since it gives every flow the
  same score;
- **FAR = 0**: it never raises a false alarm, but only because it never raises any alarm.

We score it on the validation set and on the test set.

In [2]:
# STEP A4
from sklearn.dummy import DummyClassifier

always_benign = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

baseline = pd.DataFrame({"validation": score(always_benign, X_val, y_val),
                         "test": score(always_benign, X_test, y_test)}).T
baseline.index.name = "set"

# The baseline must be exactly what the definitions say.
for part, y_part in (("validation", y_val), ("test", y_test)):
    row = baseline.loc[part]
    assert np.isclose(row["accuracy"], 1 - y_part.mean())          # = share of benign flows
    assert row["recall"] == 0 and row["far"] == 0
    assert np.isclose(row["pr_auc"], y_part.mean())                # = share of attacks

baseline.to_csv(TABLES / "A4_baseline.csv", float_format="%.4f")
print(f"Always benign, test set: accuracy {baseline.loc['test', 'accuracy']:.3f} "
      f"(= {int((y_test == 0).sum()):,} benign of {len(y_test):,} flows), recall 0, "
      f"macro-F1 {baseline.loc['test', 'macro_f1']:.3f}, PR-AUC {baseline.loc['test', 'pr_auc']:.3f}, FAR 0")
baseline.round(4)

Always benign, test set: accuracy 0.737 (= 44,969 benign of 61,021 flows), recall 0, macro-F1 0.424, PR-AUC 0.263, FAR 0


,accuracy,macro_f1,recall,pr_auc,far
set,,,,,
validation,0.7369,0.4243,0.0,0.2631,0.0
test,0.7369,0.4243,0.0,0.2631,0.0
